# agg_instance notebook walkthrough

This notebook demonstrates the four-cell workflow supported by `revisions.new_src.agg_instance`.
Each section mirrors the helper functions exported by that module so you can lift the cells into your own experiments.


## 0. Imports & toy data
We create a tiny synthetic dataset with two classes, a simple GCN explainee, and a helper for training/evaluation. Replace this block with your real dataset/model.


In [ ]:
import random
from typing import List

import torch
import torch.nn.functional as F
from torch import nn
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, global_mean_pool

from revisions.new_src.agg_instance import (
    AggregationResult,
    aggregate_instance_explanations,
    build_explainer,
    plot_eval,
    run_eval_summary,
)

# -- reproducibility ---------------------------------------------------------
torch.manual_seed(0)
random.seed(0)


def ring_edges(num_nodes: int) -> torch.Tensor:
    edges = []
    for i in range(num_nodes):
        j = (i + 1) % num_nodes
        edges.append((i, j))
        edges.append((j, i))
    return torch.tensor(edges, dtype=torch.long).t().contiguous()


def ladder_edges(num_nodes: int) -> torch.Tensor:
    edges = []
    for i in range(num_nodes - 1):
        edges.append((i, i + 1))
        edges.append((i + 1, i))
        edges.append((i, num_nodes + i))
        edges.append((num_nodes + i, i))
        edges.append((num_nodes + i, num_nodes + i + 1))
        edges.append((num_nodes + i + 1, num_nodes + i))
    return torch.tensor(edges, dtype=torch.long).t().contiguous()


def make_graph(class_id: int, noise: float = 0.0) -> Data:
    if class_id == 0:
        num_nodes = 5
        edge_index = ring_edges(num_nodes)
    else:
        num_nodes = 6
        edge_index = ladder_edges(num_nodes // 2 + 1)
        num_nodes = int(edge_index.max().item() + 1)

    x = torch.zeros((num_nodes, 2), dtype=torch.float32)
    x[:, class_id] = 1.0

    if noise > 0:
        jitter = torch.randn_like(x) * noise
        x = (x + jitter).clamp(0.0, 1.0)

    y = torch.tensor([class_id], dtype=torch.long)
    return Data(x=x, edge_index=edge_index, y=y)


def build_dataset(num_graphs_per_class: int = 32) -> List[Data]:
    dataset = []
    for cls in (0, 1):
        for _ in range(num_graphs_per_class):
            dataset.append(make_graph(cls, noise=0.05))
    random.shuffle(dataset)
    return dataset


class TinyGCN(nn.Module):
    def __init__(self, hidden_dim: int = 32):
        super().__init__()
        self.conv1 = GCNConv(2, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.lin = nn.Linear(hidden_dim, 2)

    def forward(self, batch):
        x, edge_index = batch.x.float(), batch.edge_index
        batch_vec = getattr(batch, "batch", torch.zeros(x.size(0), dtype=torch.long))
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))
        x = global_mean_pool(x, batch_vec)
        logits = self.lin(x)
        probs = logits.softmax(dim=-1)
        return {"logits": logits, "probs": probs}


def train(model: nn.Module, dataset: List[Data], epochs: int = 200) -> None:
    loader = DataLoader(dataset, batch_size=16, shuffle=True)
    opt = torch.optim.Adam(model.parameters(), lr=0.01)
    for epoch in range(epochs):
        for batch in loader:
            opt.zero_grad()
            out = model(batch)
            loss = F.cross_entropy(out["logits"], batch.y)
            loss.backward()
            opt.step()


def evaluate(model: nn.Module, dataset: List[Data]) -> float:
    loader = DataLoader(dataset, batch_size=32)
    correct = total = 0
    model.eval()
    with torch.inference_mode():
        for batch in loader:
            probs = model(batch)["probs"]
            pred = probs.argmax(dim=-1)
            correct += int((pred == batch.y).sum())
            total += batch.y.numel()
    model.train()
    return correct / max(1, total)


dataset = build_dataset()
model = TinyGCN()
train(model, dataset)
accuracy = evaluate(model, dataset)
print(f"Training accuracy: {accuracy:.3f}")


## 1. Train explainee + configure `torch_geometric.explain.Explainer`
`build_explainer` wraps the explainee with the probability adapter and lets you choose any algorithm registered inside `agg_instance`.


In [ ]:
explainer = build_explainer(
    model,
    algorithm="gnnexplainer",
    algorithm_kwargs={"epochs": 30},
    explanation_type="model",
    node_mask_type="object",
    edge_mask_type="object",
)
print(explainer)


## 2. Aggregate instance explanations into motif batches
Use `aggregate_instance_explanations` with any strategy (here `wl_topk`) to cluster per-graph explanations into canonical PyG `Batch` objects keyed by class ID.


In [ ]:
aggregation = aggregate_instance_explanations(
    dataset,
    explainer,
    explainee=model,
    strategy="wl_topk",
    strategy_kwargs={"top_p": 0.3, "wl_hops": 2},
)

for cls, batch in aggregation.by_class.items():
    print(f"Class {cls}: {batch.num_graphs} motifs, {batch.num_nodes} nodes total")


## 3. Compute the evaluation summary metrics
`run_eval_summary` plugs aggregated motifs plus observed graphs into `eval_summary`. We create dummy distance modules so the cell is self-contained; swap in your pretrained GED/omega models in practice.


In [ ]:
from torch_geometric.data import Batch as GeoBatch

class DummyDistance(nn.Module):
    def __init__(self, bias: float):
        super().__init__()
        self.register_buffer("bias", torch.tensor(bias, dtype=torch.float32))

    def evaluate(self, batch: GeoBatch):
        num_graphs = batch.num_graphs
        return self.bias.new_full((num_graphs,), float(self.bias))


dist_to_0 = DummyDistance(0.2)
dist_to_1 = DummyDistance(0.8)

obs_class_0 = [g for g in dataset if int(g.y.item()) == 0]
obs_class_1 = [g for g in dataset if int(g.y.item()) == 1]

summary_score = run_eval_summary(
    model,
    aggregation,
    observed_class_0=obs_class_0,
    observed_class_1=obs_class_1,
    dist_to_0=dist_to_0,
    dist_to_1=dist_to_1,
)
print(f"Eval summary score: {summary_score:.3f}")


## 4. Plot generated vs observed graphs
`plot_eval` visualises motif batches and reference graphs, showing the explainee confidence and GED-style distance. Provide any module compatible with `neural_approx_ged_dist`.


In [ ]:
import types

class DummyGED(nn.Module):
    def forward(self, gen_batch, obs_batch):
        num = gen_batch.num_graphs
        device = gen_batch.x.device if getattr(gen_batch, "x", None) is not None else torch.device("cpu")
        return torch.rand(num, device=device)

    def evaluate(self, gen_batch):
        num = gen_batch.num_graphs
        device = gen_batch.x.device if getattr(gen_batch, "x", None) is not None else torch.device("cpu")
        return torch.rand(num, device=device)


dataset_meta = types.SimpleNamespace(
    NODE_COLOR={0: "#94c5f8", 1: "#f59f9f"},
    NODE_CLS={0: "Type A", 1: "Type B"},
    EDGE_WIDTH={0: 2, 1: 3},
)

plot_eval(
    explainee=model,
    aggregation=aggregation,
    observed_class_0=obs_class_0,
    observed_class_1=obs_class_1,
    ged_model=DummyGED(),
    dataset={0: dataset_meta, 1: dataset_meta},
    max_pairs=2,
    layout="kamada",
)
